<a href="https://colab.research.google.com/github/sandrokhizanishvili/AML_GNN_GMA/blob/main/baseline_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Baseline GNN Model for Anti-Money Laundering

## Overview

This notebook implements three baseline GNN architectures for **edge-level transaction classification** on the IBM AML dataset (LI-Small), following the methodology of:

> Altman et al. (2023). *Realistic Synthetic Financial Transactions for Anti-Money Laundering Models.* arXiv:2306.16424

### Task
**Edge classification** — each transaction (edge) is labelled `0` (legitimate) or `1` (laundering).

### Graph Setup
| Item | Value |
|------|-------|
| Nodes (accounts) | 712,684 |
| Edges (transactions) | 6,924,049 |
| Node feature dim | 5 |
| Edge feature dim | 16 |
| Laundering rate | ~0.05% (severe imbalance) |
| Split | 60 / 20 / 20 temporal |

### Models
| Model | Description |
|-------|-------------|
| **GINe** | Graph Isomorphism Network with edge features |
| **GIN+EU** | GINe extended with explicit edge updates per layer |
| **PNA** | Principal Neighbourhood Aggregation — multiple aggregators + degree scalers |

---

## 1. Installation

Run this cell only once per Colab session, then **restart the runtime** before continuing.

In [1]:
print("🚀 SWITCHING TO PYTORCH 2.8 (FAST MODE)...")

# 1. Uninstall the current mismatching versions
# We remove the one you just spent 15 mins compiling, because re-installing
# the CORRECT version via wheels will take only 30 seconds.
# os.system("pip uninstall -y torch torchvision torchaudio torch-scatter torch-sparse pyg_lib")
!pip uninstall -y torch torchvision torchaudio torch-scatter torch-sparse pyg_lib

# 2. Install PyTorch 2.8.0 (with CUDA 12.6 support)
# We specify the version explicitly to match the PyG documentation you found.
print("⬇️ Installing PyTorch 2.8.0...")
# os.system("pip install torch==2.8.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126")
!pip install torch==2.8.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

# 3. Install Graph Libraries for PyTorch 2.8
# This link matches the table in your screenshot: torch-2.8.0 + cu126
print("⬇️ Installing Graph Libraries (Wheels)...")
# os.system("pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.8.0+cu126.html")
!pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-2.8.0+cu126.html

print("="*40)
print("✅ SETUP COMPLETE.")
print("⚠️ YOU MUST RESTART THE RUNTIME NOW (Runtime -> Restart Session)")
print("="*40)




import torch

try:
    import torch_sparse
    sparse_status = "✅ Installed"
    sparse_version = torch_sparse.__version__
except ImportError:
    sparse_status = "❌ Not Found"
    sparse_version = "N/A"

print(f"PyTorch Version:      {torch.__version__}")
print(f"CUDA Available:       {torch.cuda.is_available()}")
print(f"Torch Sparse Status:  {sparse_status} ({sparse_version})")

if torch.cuda.is_available() and sparse_status == "✅ Installed":
    print("\nSUCCESS! You are ready to run the training loop.")
else:
    print("\n⚠️ Something is still missing. Did you Restart the Runtime?")



!pip install torch_geometric

🚀 SWITCHING TO PYTORCH 2.8 (FAST MODE)...
Found existing installation: torch 2.10.0+cpu
Uninstalling torch-2.10.0+cpu:
  Successfully uninstalled torch-2.10.0+cpu
Found existing installation: torchvision 0.25.0+cpu
Uninstalling torchvision-0.25.0+cpu:
  Successfully uninstalled torchvision-0.25.0+cpu
Found existing installation: torchaudio 2.10.0+cpu
Uninstalling torchaudio-2.10.0+cpu:
  Successfully uninstalled torchaudio-2.10.0+cpu
⬇️ Installing PyTorch 2.8.0...
Looking in indexes: https://download.pytorch.org/whl/cu126
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 85.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 72.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 68.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 17.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 32.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 46.8 MB/s 

## 2. Imports & Reproducibility

In [2]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import time
import math
import random
import warnings
from google.colab import drive
warnings.filterwarnings('ignore')

# ── Numerical ─────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, precision_recall_curve, average_precision_score
)

# ── PyTorch ───────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

# ── PyTorch Geometric ─────────────────────────────────────────────────────────
import torch_geometric
from torch_geometric.data import Data
from torch_geometric.loader import LinkNeighborLoader
from torch_geometric.nn import GINEConv, PNAConv
from torch_geometric.utils import degree

# ── Progress bar ──────────────────────────────────────────────────────────────
from tqdm import tqdm

print(f'PyTorch          : {torch.__version__}')
print(f'PyTorch Geometric: {torch_geometric.__version__}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device           : {device}')

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PyTorch          : 2.8.0+cu126
PyTorch Geometric: 2.7.0
Device           : cpu


## 3. Load Graph Snapshots

Three PyG `Data` objects saved by `Data_prepration.ipynb`:
- `train_graph` — training edges only, all evaluated
- `val_graph` — train + val edges, evaluated on val portion only
- `test_graph` — all edges, evaluated on test portion only

This cumulative snapshot design ensures val/test edges have access to historical context for message passing, matching the paper's protocol.

In [4]:
drive.mount('/content/drive')

os.chdir('/content/drive/MyDrive/GMA_GNN_AML')

Mounted at /content/drive


In [5]:
DATA_DIR = 'Data'

# weights_only=False required because PyG Data objects contain custom classes
# that cannot be loaded with PyTorch 2.6+ default weights_only=True
train_graph = torch.load(os.path.join(DATA_DIR, 'train_graph.pt'), weights_only=False)
val_graph   = torch.load(os.path.join(DATA_DIR, 'val_graph.pt'),   weights_only=False)
test_graph  = torch.load(os.path.join(DATA_DIR, 'test_graph.pt'),  weights_only=False)

def describe_graph(g, name):
    """Print a summary of a graph snapshot."""
    labels   = g.y[g.eval_mask]
    n_pos    = (labels == 1).sum().item()
    n_eval   = g.eval_mask.sum().item()
    print(f'{name}:')
    print(f'  Nodes          : {g.num_nodes:,}')
    print(f'  Edges (total)  : {g.edge_index.shape[1]:,}')
    print(f'  Eval edges     : {n_eval:,}')
    print(f'  Laundering     : {n_pos:,}  ({100*n_pos/n_eval:.4f}%)')
    print(f'  Node feat dim  : {g.x.shape[1]}')
    print(f'  Edge feat dim  : {g.edge_attr.shape[1]}')
    print()

describe_graph(train_graph, 'train_graph')
describe_graph(val_graph,   'val_graph')
describe_graph(test_graph,  'test_graph')

# ── Label sanity check ────────────────────────────────────────────────────────
for g, name in [(train_graph, 'train'), (val_graph, 'val'), (test_graph, 'test')]:
    labels = g.y[g.eval_mask]
    assert (labels == -1).sum() == 0, f'{name}: -1 labels found in eval set!'
    assert labels.unique().tolist() == [0, 1] or set(labels.unique().tolist()) <= {0, 1}
print('Label sanity check passed — no -1 values in eval masks.')

train_graph:
  Nodes          : 712,684
  Edges (total)  : 4,154,429
  Eval edges     : 4,154,429
  Laundering     : 1,813  (0.0436%)
  Node feat dim  : 5
  Edge feat dim  : 16

val_graph:
  Nodes          : 712,684
  Edges (total)  : 5,539,239
  Eval edges     : 1,384,810
  Laundering     : 827  (0.0597%)
  Node feat dim  : 5
  Edge feat dim  : 16

test_graph:
  Nodes          : 712,684
  Edges (total)  : 6,924,049
  Eval edges     : 1,384,810
  Laundering     : 925  (0.0668%)
  Node feat dim  : 5
  Edge feat dim  : 16

Label sanity check passed — no -1 values in eval masks.


## 4. Hyperparameters

In [6]:
# ── Dimensions (inferred from data) ──────────────────────────────────────────
NODE_DIM = train_graph.x.shape[1]          # 5
EDGE_DIM = train_graph.edge_attr.shape[1]  # 16

# ── Model ─────────────────────────────────────────────────────────────────────
HIDDEN_DIM = 64    # matches the paper: hidden embedding size 64
NUM_LAYERS = 2     # matches the paper: 2 GNN layers
DROPOUT    = 0.3

# ── Training ──────────────────────────────────────────────────────────────────
EPOCHS       = 10
LR           = 1e-3
WEIGHT_DECAY = 1e-5

# ── Neighbourhood sampling ────────────────────────────────────────────────────
# num_neighbors[0] = neighbours to sample at 2-hop (outermost)
# num_neighbors[1] = neighbours to sample at 1-hop (closest to seed)
# Must have one value per GNN layer
NUM_NEIGHBORS = [100, 100]
BATCH_SIZE    = 16384 # 4096

# ── Class imbalance ───────────────────────────────────────────────────────────
# After oversampling to 20% positives, effective ratio = 4:1
# pos_weight = 4.0 balances the gradient signal between classes
n_neg_train = (train_graph.y[train_graph.eval_mask] == 0).sum().item()
n_pos_train = (train_graph.y[train_graph.eval_mask] == 1).sum().item()
print(f'Train imbalance  : {n_neg_train/n_pos_train:.0f}:1  '
      f'({n_pos_train:,} pos / {n_neg_train:,} neg)')

POS_WEIGHT = torch.tensor([4.0], device=device)
criterion  = nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT)

print(f'\nConfig:')
print(f'  NODE_DIM={NODE_DIM}, EDGE_DIM={EDGE_DIM}')
print(f'  HIDDEN_DIM={HIDDEN_DIM}, NUM_LAYERS={NUM_LAYERS}, DROPOUT={DROPOUT}')
print(f'  EPOCHS={EPOCHS}, LR={LR}, BATCH_SIZE={BATCH_SIZE}')
print(f'  NUM_NEIGHBORS={NUM_NEIGHBORS}')
print(f'  POS_WEIGHT={POS_WEIGHT.item()}')

Train imbalance  : 2290:1  (1,813 pos / 4,152,616 neg)

Config:
  NODE_DIM=5, EDGE_DIM=16
  HIDDEN_DIM=64, NUM_LAYERS=2, DROPOUT=0.3
  EPOCHS=10, LR=0.001, BATCH_SIZE=16384
  NUM_NEIGHBORS=[100, 100]
  POS_WEIGHT=4.0


## 5. Model Architectures

### Encode → Decode Pattern

`LinkNeighborLoader` separates two sets of edges per batch:

- `batch.edge_index` — **context edges** used for message passing (building node embeddings)
- `batch.edge_label_index` — **seed edges** that we actually want to classify

This requires an explicit encode → decode architecture:
```
encode: x, edge_index, edge_attr  →  node embeddings h
decode: h, edge_label_index        →  logits for seed edges
```


### Why LayerNorm Instead of BatchNorm

BatchNorm normalises across the batch dimension. With ~200,000 context edges per batch and only ~800 laundering edges (0.4%), the batch statistics are dominated entirely by legitimate transactions — effectively normalising away the laundering signal. LayerNorm normalises per node across the feature dimension, which is safe regardless of class distribution.

In [7]:
def build_mlp(in_dim, hidden_dim, out_dim, num_layers=2, dropout=0.3):
    """
    Builds a multi-layer perceptron with LayerNorm and Dropout.
    Used as the update function inside GNN layers and as the final classifier.
    """
    layers = []
    dims   = [in_dim] + [hidden_dim] * (num_layers - 1) + [out_dim]
    for i in range(len(dims) - 1):
        layers.append(nn.Linear(dims[i], dims[i+1]))
        if i < len(dims) - 2:   # no activation/norm on the final output layer
            layers.append(nn.ReLU())
            layers.append(nn.LayerNorm(dims[i+1]))
            layers.append(nn.Dropout(dropout))
    return nn.Sequential(*layers)

In [ ]:
'''
GINEConv (handles graph structure):
┌──────────────────────────────────────────────────┐
│  For each node v:                                │
│    agg = SUM[ ReLU(h[u] + e_{uv}) ]             │
│    input = h[v] + agg                            │
│                                                  │
│    YOUR mlp(input):  ◄── build_mlp lives here   │
│    ┌────────────────────────────────────────┐    │
│    │ Linear(64→128) → ReLU → LN → Dropout  │    │
│    │ → Linear(128→64)                       │    │
│    └────────────────────────────────────────┘    │
│                                                  │
│    h_new[v] = mlp output                        │
└──────────────────────────────────────────────────┘
'''

### Model 1 — GINe (Graph Isomorphism Network with Edge Features)

In [8]:
class GINe(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_dim, num_layers, dropout=0.3):
        super().__init__()
        self.dropout   = dropout
        self.node_proj = nn.Linear(node_dim, hidden_dim) # INPUT → h projection
        self.edge_proj = nn.Linear(edge_dim, hidden_dim) # INPUT → e projection

        self.convs = nn.ModuleList() # 2 GNN layers
        self.norms = nn.ModuleList() # one norm per layer
        for _ in range(num_layers):
            mlp = build_mlp(hidden_dim, hidden_dim * 2, hidden_dim, dropout=dropout)
            self.convs.append(GINEConv(mlp, edge_dim=hidden_dim))
            self.norms.append(nn.LayerNorm(hidden_dim))

        # ── 192-dim input: h[src](64) + h[dst](64) + e(64) ───────
        self.edge_classifier = build_mlp(
            in_dim     = hidden_dim * 3,
            hidden_dim = hidden_dim,
            out_dim    = 1,
            dropout    = dropout,
        )

    def encode(self, x, edge_index, edge_attr):
        """Message passing over context edges → node embeddings."""
        h = F.relu(self.node_proj(x)) # x [N,5]  → h⁰ [N,64]
        e = F.relu(self.edge_proj(edge_attr)) # edge_attr [E,16] → e [E,64]

        for conv, norm in zip(self.convs, self.norms):
            h = conv(h, edge_index, e) # GINEConv: SUM ReLU(h[u] + e_{uv})
            h = norm(h)                # LayerNorm per node
            h = F.relu(h)
            h = F.dropout(h, p=self.dropout, training=self.training)
        return h

    def decode(self, h, edge_label_index, edge_label_attr):
        """
        Classify seed edges using node embeddings AND seed edge features.

        Parameters
        ----------
        h                : node embeddings from encode()  [N, 64]
        edge_label_index : seed edge endpoints            [2, n_seeds]
        edge_label_attr  : raw features of seed edges     [n_seeds, 16]
        """
        src, dst = edge_label_index

        # Project seed edge features to hidden dim
        # (same projection as used during message passing)
        e_seed = F.relu(self.edge_proj(edge_label_attr))  # [n_seeds, 64]

        # Concatenate: sender context + receiver context + transaction features
        edge_emb = torch.cat([h[src], h[dst], e_seed], dim=-1)  # [n_seeds, 192]
        return self.edge_classifier(edge_emb).squeeze(-1)

    def forward(self, x, edge_index, edge_attr, edge_label_index, edge_label_attr):
        h = self.encode(x, edge_index, edge_attr)
        return self.decode(h, edge_label_index, edge_label_attr)

## 6. Data Loader

### Why Oversample During Training Only

With 0.05% laundering, each batch of 4096 edges contains on average **2 laundering transactions**. With pos_weight=4, the gradient from 2 positives (×4 weight) is still overwhelmed by 4094 negatives — the model collapses to predicting everything as legitimate.

By repeating positive seed edges ~418× during training, we raise the positive ratio to 20%, so each batch contains ~800 laundering edges — enough gradient signal to actually learn.

Evaluation always uses the **real distribution** (no oversampling) to give honest metrics.

In [12]:
# def make_loader(graph, shuffle=True, verbose=False):
#     """Loader without oversampling — uses real class distribution."""
#     seed_edge_index = graph.edge_index[:, graph.eval_mask]
#     seed_labels     = graph.y[graph.eval_mask].float()

#     if verbose:
#       n_pos_after = (seed_labels == 1).sum().item()
#       n_neg_after = (seed_labels == 0).sum().item()
#       print(f'  Data: {n_pos_after:,} pos '
#             f'({100*n_pos_after/(n_pos_after+n_neg_after):.4f}%) '
#             f'| {n_neg_after:,} neg')


#     return LinkNeighborLoader(
#         data             = graph,
#         num_neighbors    = NUM_NEIGHBORS,
#         edge_label_index = seed_edge_index,
#         edge_label       = seed_labels,
#         batch_size       = BATCH_SIZE,
#         shuffle          = shuffle,
#         num_workers      = 0,
#         pin_memory       = False,
#     )

def make_loader(graph, shuffle=True, verbose=False):
    seed_mask       = graph.eval_mask
    seed_edge_index = graph.edge_index[:, seed_mask]
    seed_labels     = graph.y[seed_mask].float()
    seed_edge_attr  = graph.edge_attr[seed_mask]   # ← grab features too

    if verbose:
        n_pos = (seed_labels == 1).sum().item()
        n_neg = (seed_labels == 0).sum().item()
        print(f'  Seed edges : {n_pos + n_neg:,} total')
        print(f'  Laundering : {n_pos:,} ({100*n_pos/(n_pos+n_neg):.4f}%)')

    return LinkNeighborLoader(
        data             = graph,
        num_neighbors    = NUM_NEIGHBORS,
        edge_label_index = seed_edge_index,
        edge_label       = seed_labels,
        batch_size       = BATCH_SIZE,
        shuffle          = shuffle,
        num_workers      = 0,
        pin_memory       = False,
    ), seed_edge_attr   # return alongside loader

## 7. Training & Evaluation

In [15]:
# Higher pos_weight to compensate for no oversampling
# Real ratio ~2290:1 → sqrt gives ~48, we use 50 as a round number
# criterion = nn.BCEWithLogitsLoss(
#     pos_weight=torch.tensor([50.0], device=device)
# )

# def train_epoch(model, graph, optimizer):
#     """Training epoch without oversampling."""
#     model.train()
#     result     = make_loader(graph, shuffle=True, verbose=True)
#     loader = result[0] if isinstance(result, tuple) else result
#     total_loss = 0.0
#     n_batches  = 0
#     pbar       = tqdm(loader, desc='  Training (no oversample)', leave=False)

#     for batch in pbar:
#         batch  = batch.to(device)
#         optimizer.zero_grad()
#         logits = model(
#             batch.x, batch.edge_index,
#             batch.edge_attr, batch.edge_label_index,
#         )
#         loss = criterion(logits, batch.edge_label)
#         loss.backward()
#         nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
#         optimizer.step()
#         total_loss += loss.item()
#         n_batches  += 1
#         pbar.set_postfix({'loss': f'{loss.item():.4f}'})

#     return total_loss / max(n_batches, 1)


# @torch.no_grad()
# def evaluate(model, graph, threshold=0.5):
#     """
#     Evaluate model on a graph snapshot.

#     Always uses the real class distribution (no oversampling) for honest metrics.
#     Threshold=0.5 is the default — tune only AFTER training is complete
#     using find_best_threshold() on the validation set.

#     Returns dict with: f1, precision, recall, auc
#     """
#     model.eval()
#     loader     = make_loader(graph, shuffle=False)
#     all_probs  = []
#     all_labels = []
#     pbar       = tqdm(loader, desc='  Validation', leave=False)

#     for batch in pbar:
#         batch  = batch.to(device)
#         logits = model(
#             batch.x,
#             batch.edge_index,
#             batch.edge_attr,
#             batch.edge_label_index,
#         )
#         all_probs.append(torch.sigmoid(logits).cpu().numpy())
#         all_labels.append(batch.edge_label.long().cpu().numpy())

#     all_probs  = np.concatenate(all_probs)
#     all_labels = np.concatenate(all_labels)
#     preds      = (all_probs >= threshold).astype(int)

#     f1  = f1_score(all_labels, preds, pos_label=1, zero_division=0)
#     pre = precision_score(all_labels, preds, pos_label=1, zero_division=0)
#     rec = recall_score(all_labels, preds, pos_label=1, zero_division=0)
#     try:
#         auc = roc_auc_score(all_labels, all_probs)
#     except ValueError:
#         auc = float('nan')

#     return {'f1': f1, 'precision': pre, 'recall': rec, 'auc': auc}


# def run_training(model, model_name, checkpoint_path, epochs=EPOCHS):
#     """Training loop without oversampling — ablation study."""
#     optimizer = Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
#     scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)

#     best_val_auc = 0.0
#     best_state   = None
#     history      = []

#     print(f'\n{"="*60}')
#     print(f'Training {model_name} — NO OVERSAMPLING (ablation)')
#     print(f'  Parameters : {sum(p.numel() for p in model.parameters()):,}')
#     print(f'  pos_weight : 50.0 (compensates for no oversampling)')
#     print(f'{"="*60}')

#     for epoch in range(1, epochs + 1):
#         print(f'\n--- Epoch {epoch}/{epochs} ---')
#         t0          = time.time()
#         train_loss  = train_epoch(model, train_graph, optimizer)
#         scheduler.step()
#         val_metrics = evaluate(model, val_graph)
#         elapsed     = time.time() - t0

#         history.append({'epoch': epoch, 'loss': train_loss, **val_metrics})

#         improved = ''
#         if val_metrics['auc'] > best_val_auc:
#             best_val_auc = val_metrics['auc']
#             best_state   = {k: v.clone() for k, v in model.state_dict().items()}
#             torch.save(model.state_dict(), checkpoint_path)
#             improved     = '  --> New Best Model!'

#         print(
#             f'Result: Train Loss: {train_loss:.4f} | '
#             f'Val F1: {val_metrics["f1"]:.4f} | '
#             f'Val Pre: {val_metrics["precision"]:.4f} | '
#             f'Val Rec: {val_metrics["recall"]:.4f} | '
#             f'Val AUC: {val_metrics["auc"]:.4f} | '
#             f'Time: {elapsed:.1f}s'
#             f'{improved}'
#         )

#     if best_state is not None:
#         model.load_state_dict(best_state)

#     test_metrics = evaluate(model, test_graph)
#     print(f'\n{"="*60}')
#     print(f'Final Test Results — {model_name} (no oversampling)')
#     print(f'  F1        : {test_metrics["f1"]:.4f}')
#     print(f'  Precision : {test_metrics["precision"]:.4f}')
#     print(f'  Recall    : {test_metrics["recall"]:.4f}')
#     print(f'  AUC-ROC   : {test_metrics["auc"]:.4f}')
#     print(f'{"="*60}')

#     return model, history, test_metrics

In [17]:
criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([50.0], device=device)
)



def train_epoch(model, graph, optimizer):
    model.train()
    loader, seed_edge_attr = make_loader(graph, shuffle=True, verbose=True)
    total_loss = 0.0
    n_batches  = 0
    pbar       = tqdm(loader, desc='  Training', leave=False)

    for batch in pbar:
        batch = batch.to(device)
        optimizer.zero_grad()

        # Get seed edge features for this batch using input_id
        # batch.input_id contains the indices of seed edges in the original
        # seed pool — use them to look up the correct edge features
        seed_attr = seed_edge_attr[batch.input_id].to(device)

        logits = model(
            batch.x,
            batch.edge_index,
            batch.edge_attr,
            batch.edge_label_index,
            seed_attr,               # ← seed edge raw features
        )

        loss = criterion(logits, batch.edge_label)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        n_batches  += 1
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    return total_loss / max(n_batches, 1)


@torch.no_grad()
def evaluate(model, graph, threshold=0.5):
    model.eval()
    loader, seed_edge_attr = make_loader(graph, shuffle=False)
    all_probs  = []
    all_labels = []
    pbar       = tqdm(loader, desc='  Validation', leave=False)

    for batch in pbar:
        batch     = batch.to(device)
        seed_attr = seed_edge_attr[batch.input_id].to(device)

        logits = model(
            batch.x,
            batch.edge_index,
            batch.edge_attr,
            batch.edge_label_index,
            seed_attr,
        )
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(batch.edge_label.long().cpu().numpy())

    all_probs  = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    preds      = (all_probs >= threshold).astype(int)

    f1  = f1_score(all_labels, preds, pos_label=1, zero_division=0)
    pre = precision_score(all_labels, preds, pos_label=1, zero_division=0)
    rec = recall_score(all_labels, preds, pos_label=1, zero_division=0)
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except ValueError:
        auc = float('nan')

    return {'f1': f1, 'precision': pre, 'recall': rec, 'auc': auc}


def run_training(model, model_name, checkpoint_path, epochs=EPOCHS):
    """Training loop without oversampling — ablation study."""
    optimizer = Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)

    best_val_auc = 0.0
    best_state   = None
    history      = []

    print(f'\n{"="*60}')
    print(f'Training {model_name} — NO OVERSAMPLING (ablation)')
    print(f'  Parameters : {sum(p.numel() for p in model.parameters()):,}')
    print(f'  pos_weight : 50.0 (compensates for no oversampling)')
    print(f'{"="*60}')

    for epoch in range(1, epochs + 1):
        print(f'\n--- Epoch {epoch}/{epochs} ---')
        t0          = time.time()
        train_loss  = train_epoch(model, train_graph, optimizer)
        scheduler.step()
        val_metrics = evaluate(model, val_graph)
        elapsed     = time.time() - t0

        history.append({'epoch': epoch, 'loss': train_loss, **val_metrics})

        improved = ''
        if val_metrics['auc'] > best_val_auc:
            best_val_auc = val_metrics['auc']
            best_state   = {k: v.clone() for k, v in model.state_dict().items()}
            torch.save(model.state_dict(), checkpoint_path)
            improved     = '  --> New Best Model!'

        print(
            f'Result: Train Loss: {train_loss:.4f} | '
            f'Val F1: {val_metrics["f1"]:.4f} | '
            f'Val Pre: {val_metrics["precision"]:.4f} | '
            f'Val Rec: {val_metrics["recall"]:.4f} | '
            f'Val AUC: {val_metrics["auc"]:.4f} | '
            f'Time: {elapsed:.1f}s'
            f'{improved}'
        )

    if best_state is not None:
        model.load_state_dict(best_state)

    test_metrics = evaluate(model, test_graph)
    print(f'\n{"="*60}')
    print(f'Final Test Results — {model_name} (no oversampling)')
    print(f'  F1        : {test_metrics["f1"]:.4f}')
    print(f'  Precision : {test_metrics["precision"]:.4f}')
    print(f'  Recall    : {test_metrics["recall"]:.4f}')
    print(f'  AUC-ROC   : {test_metrics["auc"]:.4f}')
    print(f'{"="*60}')

    return model, history, test_metrics

## 8. Train All Three Models

In [ ]:
torch.manual_seed(SEED)
gin_model = GINe(
    node_dim   = NODE_DIM,
    edge_dim   = EDGE_DIM,
    hidden_dim = HIDDEN_DIM,
    num_layers = NUM_LAYERS,
    dropout    = DROPOUT,
).to(device)

gin_model, gin_history, gin_test = run_training(
    gin_model, 'GINe (no oversample)',checkpoint_path = 'Models/GINe/baseline_11_05_26.pt' , epochs=EPOCHS
)


Training GINe (no oversample) — NO OVERSAMPLING (ablation)
  Parameters : 56,257
  pos_weight : 50.0 (compensates for no oversampling)

--- Epoch 1/10 ---
  Seed edges : 4,154,429 total
  Laundering : 1,813 (0.0436%)


  Training:   3%|▎         | 7/254 [00:44<28:48,  7.00s/it, loss=0.0941]